## Summary

This notebook demonstrated the complete XAI implementation for the house price prediction model:

### ✓ Implemented Features:

1. **ModelExplainer Class** - Comprehensive explainability tool supporting:
   - SHAP (SHapley Additive exPlanations) for precise feature contributions
   - LIME (Local Interpretable Model-agnostic Explanations) for local interpretability
   - Feature Importance from the trained model

2. **API Endpoints**:
   - `POST /predict` - Regular predictions
   - `POST /explain` - Comprehensive explanations with multiple methods
   - `GET /feature-importance` - Global feature importance scores

3. **Integration Components**:
   - `ExplainService` - Business logic for explanations
   - Enhanced `schemas.py` - Request/response models for XAI
   - Improved `main.py` - New FastAPI endpoints
   - Training data loading for SHAP calculation

### 📊 How to Use:

**1. Start the API Server:**
```bash
python src/api/main.py
```

**2. Make a Prediction with Explanation:**
```bash
curl -X POST "http://localhost:8000/explain" \
  -H "Content-Type: application/json" \
  -d '{
    "house_data": {...},
    "methods": ["shap", "feature_importance"]
  }'
```

**3. Get Feature Importance:**
```bash
curl -X GET "http://localhost:8000/feature-importance"
```

### 📚 Files Modified/Created:

- `src/ml_pipeline/explainability.py` - Enhanced XAI module
- `src/api/main.py` - Added XAI endpoints
- `src/api/schemas.py` - Added XAI request/response models
- `src/api/services.py` - Added ExplainService
- `requirements.txt` - Added SHAP and LIME dependencies
- `notebooks/XAI_Demo.ipynb` - This demonstration notebook

In [ ]:
# Test 5: Multiple predictions
print("\n" + "=" * 60)
print("Test 5: Performance Comparison - Multiple Explanation Requests")
print("=" * 60)

# Create different house samples
samples = [
    {"label": "Luxury House", "GrLivArea": 3500, "OverallQual": 9},
    {"label": "Average House", "GrLivArea": 1500, "OverallQual": 5},
    {"label": "Budget House", "GrLivArea": 800, "OverallQual": 3},
]

results = []

try:
    for sample_info in samples:
        # Create payload based on template
        payload = predict_payload.copy()
        payload.update({
            "GrLivArea": sample_info["GrLivArea"],
            "OverallQual": sample_info["OverallQual"]
        })
        
        explain_request = {
            "house_data": payload,
            "methods": ["feature_importance"]
        }
        
        start_time = time.time()
        response = requests.post(f"{API_URL}/explain", json=explain_request)
        elapsed_time = time.time() - start_time
        
        if response.status_code == 200:
            result = response.json()
            results.append({
                "label": sample_info["label"],
                "prediction": result.get('prediction'),
                "response_time": elapsed_time,
                "status": "✓ Success"
            })
        else:
            results.append({
                "label": sample_info["label"],
                "prediction": "N/A",
                "response_time": elapsed_time,
                "status": f"✗ Error {response.status_code}"
            })
except requests.exceptions.ConnectionError:
    print("✗ Could not connect to API")
    results = []
except Exception as e:
    print(f"✗ Error: {e}")
    results = []

# Display results
if results:
    print("\nResults:")
    for result in results:
        print(f"  {result['label']:20s} | Price: ${result['prediction']:>12,.2f} | Time: {result['response_time']:>6.2f}s | {result['status']}")

In [ ]:
# Test 4: Feature Importance endpoint
print("\n" + "=" * 60)
print("Test 4: Feature Importance Endpoint")
print("=" * 60)

try:
    response = requests.get(f"{API_URL}/feature-importance")
    print(f"Status Code: {response.status_code}")
    
    if response.status_code == 200:
        importance = response.json()
        print(f"\nModel Type: {importance.get('model_type', 'N/A')}")
        print(f"Method: {importance.get('method', 'N/A')}")
        print(f"\nTop 10 Most Important Features:")
        
        for i, feat in enumerate(importance.get('top_10_features', [])[:10], 1):
            print(f"  {i:2d}. {feat['feature']:20s} : {feat['importance']:.6f}")
    else:
        print(f"Error: {response.text}")
        
except requests.exceptions.ConnectionError:
    print("✗ Could not connect to API")
except Exception as e:
    print(f"✗ Error: {e}")

In [ ]:
# Test 3: Explain endpoint with SHAP
print("\n" + "=" * 60)
print("Test 3: Explain Endpoint (SHAP + Feature Importance)")
print("=" * 60)

explain_payload = {
    "house_data": predict_payload,
    "methods": ["shap", "feature_importance"]
}

try:
    start_time = time.time()
    response = requests.post(f"{API_URL}/explain", json=explain_payload)
    elapsed_time = time.time() - start_time
    
    print(f"Status Code: {response.status_code}")
    print(f"Response Time: {elapsed_time:.2f}s")
    
    if response.status_code == 200:
        explanation = response.json()
        print(f"\nPrediction: ${explanation.get('prediction', 'N/A'):,.2f}")
        
        # Display available explanation methods
        methods = explanation.get('explanation_methods', {})
        print(f"\nAvailable Explanation Methods: {list(methods.keys())}")
        
        # Display SHAP explanation
        if 'shap' in methods and 'error' not in methods['shap']:
            shap_exp = methods['shap']
            print(f"\nSHAP Explanation:")
            print(f"  - Base Value: ${shap_exp.get('base_value', 'N/A'):,.2f}")
            print(f"  - Top 5 Features:")
            for i, feat in enumerate(shap_exp.get('top_5_features', [])[:5], 1):
                print(f"      {i}. {feat['feature']}: {feat['shap_value']:,.2f}")
        
        # Display Feature Importance
        if 'feature_importance' in methods and 'error' not in methods['feature_importance']:
            imp_exp = methods['feature_importance']
            print(f"\nFeature Importance:")
            print(f"  - Top 5 Features:")
            for i, feat in enumerate(imp_exp.get('top_10_features', [])[:5], 1):
                print(f"      {i}. {feat['feature']}: {feat['importance']:.6f}")
    else:
        print(f"Error: {response.text}")
        
except requests.exceptions.ConnectionError:
    print("✗ Could not connect to API")
except Exception as e:
    print(f"✗ Error: {e}")
    print(f"Response: {response.text if 'response' in locals() else 'N/A'}")

In [ ]:
# Test 2: Prediction endpoint
print("\n" + "=" * 60)
print("Test 2: Regular Prediction")
print("=" * 60)

predict_payload = {
    "LotArea": 8450,
    "OverallQual": 7,
    "OverallCond": 5,
    "YearBuilt": 2003,
    "YearRemodAdd": 2003,
    "TotalBsmtSF": 1000,
    "FirstFlrSF": 856,
    "SecondFlrSF": 854,
    "GrLivArea": 1710,
    "FullBath": 2,
    "HalfBath": 1,
    "BsmtFullBath": 1,
    "BsmtHalfBath": 0,
    "Bedroom": 3,
    "Kitchen": 1,
    "TotRmsAbvGrd": 8,
    "Fireplaces": 1,
    "GarageCars": 2,
    "GarageSF": 548,
    "MSZoning": "RL",
    "Neighborhood": "NAmes",
    "BldgType": "1Fam",
    "ExterQual": "Gd",
    "ExterCond": "TA",
    "BsmtQual": "Gd",
    "BsmtCond": "TA",
    "HeatingQC": "Ex",
    "KitchenQual": "Gd",
    "FireplaceQu": "Ta",
    "GarageQual": "TA",
    "GarageCond": "TA",
    "MoSold": 7,
    "YrSold": 2022,
    "SaleType": "WD",
    "SaleCondition": "Normal"
}

try:
    response = requests.post(f"{API_URL}/predict", json=predict_payload)
    print(f"Status Code: {response.status_code}")
    prediction_result = response.json()
    print(f"Predicted Price: ${prediction_result.get('predicted_price', 'N/A'):,.2f}")
    print(f"Confidence: {prediction_result.get('confidence', 'N/A'):.2%}")
    print(f"Model: {prediction_result.get('model_name', 'N/A')}")
except requests.exceptions.ConnectionError:
    print("✗ Could not connect to API")
except Exception as e:
    print(f"✗ Error: {e}")

In [ ]:
import requests
import time

# API endpoint configuration
API_URL = "http://localhost:8000"  # Change if API is running on different host/port

print(f"API Base URL: {API_URL}")
print("\nTesting API endpoints...\n")

# Test 1: Health check
print("=" * 60)
print("Test 1: Health Check")
print("=" * 60)
try:
    response = requests.get(f"{API_URL}/health")
    print(f"Status Code: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
except requests.exceptions.ConnectionError:
    print("✗ Could not connect to API. Make sure the API server is running.")
    print("  Run: python src/api/main.py")

## 7. Test Explanation Endpoints with Sample Predictions

## 6. API Endpoint Overview

The `/explain` API endpoint has been integrated into the FastAPI backend and provides the following capabilities:

### Endpoint Details:
- **URL**: `POST /explain`
- **Description**: Generate explanation for a house price prediction using XAI methods
- **Supported Methods**: `shap`, `lime`, `feature_importance`

### Request Format:
```json
{
  "house_data": {
    "LotArea": 8450,
    "OverallQual": 7,
    ...
  },
  "methods": ["shap", "feature_importance"]
}
```

### Response Format:
```json
{
  "prediction": 180000.50,
  "explanation_methods": {
    "shap": {
      "method": "SHAP",
      "base_value": 200000.0,
      "feature_contributions": [...],
      "top_5_features": [...],
      "model_type": "XGBRegressor"
    },
    "feature_importance": {
      "method": "Model Feature Importance",
      "feature_importance": [...],
      "top_10_features": [...]
    }
  }
}
```

In [ ]:
# Visualize feature importance
if explainer is not None and "error" not in importance_explanation:
    # Prepare data for visualization
    top_features = importance_explanation['top_10_features'][:15]
    features = [f['feature'] for f in top_features]
    importances = [f['importance'] for f in top_features]
    
    # Create visualization
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(features, importances, color='steelblue', alpha=0.8)
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_title('Top 15 Features by Model Importance', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("✓ Feature importance chart displayed")

In [ ]:
# Get feature importance
if explainer is not None:
    print("Extracting feature importance...")
    importance_explanation = explainer.get_feature_importance()
    
    if "error" not in importance_explanation:
        print("\n✓ Feature Importance Retrieved:")
        print(f"  Model Type: {importance_explanation['model_type']}")
        print(f"\n  Top 10 Most Important Features:")
        
        for i, feature in enumerate(importance_explanation['top_10_features'][:10], 1):
            print(f"    {i}. {feature['feature']}: Importance = {feature['importance']:.6f}")
    else:
        print(f"✗ Feature importance extraction failed: {importance_explanation['error']}")

## 5. Extract Feature Importance

In [ ]:
# Generate LIME explanation
if explainer is not None:
    print("Generating LIME explanation...")
    lime_explanation = explainer.explain_lime(X_sample_processed)
    
    if "error" not in lime_explanation:
        print("\n✓ LIME Explanation Generated:")
        print(f"  Prediction: ${lime_explanation['prediction']:,.2f}")
        print(f"  Model Type: {lime_explanation['model_type']}")
        print(f"\n  Feature Contributions (Top 5):")
        
        for i, feature in enumerate(lime_explanation['feature_contributions'][:5], 1):
            print(f"    {i}. {feature['feature']}: Weight = {feature['weight']:.6f}")
    else:
        print(f"✓ LIME explanation: {lime_explanation['error']}")
        print("  (LIME may not be available in this environment)")

## 4. Generate LIME Explanations

In [ ]:
# Visualize SHAP feature contributions
if explainer is not None and "error" not in shap_explanation:
    # Prepare data for visualization
    features = [f['feature'] for f in shap_explanation['top_5_features'][:10]]
    shap_values = [f['shap_value'] for f in shap_explanation['top_5_features'][:10]]
    
    # Create visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['green' if v > 0 else 'red' for v in shap_values]
    ax.barh(features, shap_values, color=colors, alpha=0.7)
    ax.set_xlabel('SHAP Value (Impact on Prediction)', fontsize=12)
    ax.set_title('Top 10 Features by SHAP Values', fontsize=14, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    plt.tight_layout()
    plt.show()
    
    print("✓ SHAP contribution chart displayed")

In [ ]:
# Generate SHAP explanation
if explainer is not None:
    print("Generating SHAP explanation...")
    shap_explanation = explainer.explain_shap(X_sample_processed)
    
    if "error" not in shap_explanation:
        print("\n✓ SHAP Explanation Generated:")
        print(f"  Base Value: ${shap_explanation['base_value']:,.2f}")
        print(f"  Model Type: {shap_explanation['model_type']}")
        print(f"\n  Top 5 Feature Contributions:")
        
        for i, feature in enumerate(shap_explanation['top_5_features'][:5], 1):
            print(f"    {i}. {feature['feature']}: SHAP Value = {feature['shap_value']:,.2f}, Feature Value = {feature['value']:,.2f}")
    else:
        print(f"✗ SHAP explanation failed: {shap_explanation['error']}")
else:
    print("✗ Explainer not initialized")

In [ ]:
# Create a sample house instance for explanation
sample_data = {
    "LotArea": 8450,
    "OverallQual": 7,
    "OverallCond": 5,
    "YearBuilt": 2003,
    "YearRemodAdd": 2003,
    "TotalBsmtSF": 1000,
    "FirstFlrSF": 856,
    "SecondFlrSF": 854,
    "GrLivArea": 1710,
    "FullBath": 2,
    "HalfBath": 1,
    "BsmtFullBath": 1,
    "BsmtHalfBath": 0,
    "Bedroom": 3,
    "Kitchen": 1,
    "TotRmsAbvGrd": 8,
    "Fireplaces": 1,
    "GarageCars": 2,
    "GarageSF": 548,
    "MSZoning": "RL",
    "Neighborhood": "NAmes",
    "BldgType": "1Fam",
    "ExterQual": "Gd",
    "ExterCond": "TA",
    "BsmtQual": "Gd",
    "BsmtCond": "TA",
    "HeatingQC": "Ex",
    "KitchenQual": "Gd",
    "FireplaceQu": "Ta",
    "GarageQual": "TA",
    "GarageCond": "TA",
    "MoSold": 7,
    "YrSold": 2022,
    "SaleType": "WD",
    "SaleCondition": "Normal"
}

# Convert to HousePriceInput and DataFrame
house_input = HousePriceInput(**sample_data)
X_sample = pd.DataFrame([sample_data])

# Preprocess if needed
if service.preprocessor:
    try:
        X_sample_processed = service.preprocessor.transform(X_sample)
    except:
        X_sample_processed = X_sample
else:
    X_sample_processed = X_sample

print("Sample house data created")
print(f"Prediction: ${service.model.predict(X_sample_processed)[0]:,.2f}")

## 3. Generate SHAP Explanations

In [ ]:
# Initialize ModelExplainer
print("Initializing ModelExplainer...")

if service.is_ready() and X_train is not None:
    explainer = ModelExplainer(
        model=service.model,
        X_train=X_train,
        feature_names=list(X_train.columns)
    )
    print("✓ ModelExplainer initialized successfully")
    
    # Check if explainers are ready
    if explainer.shap_explainer is not None:
        print("  ✓ SHAP explainer ready")
    else:
        print("  ✗ SHAP explainer not available")
    
    if explainer.lime_explainer is not None:
        print("  ✓ LIME explainer ready")
    else:
        print("  ✗ LIME explainer not available (optional)")
else:
    print("✗ Cannot initialize explainer - model or training data not available")
    explainer = None

## 2. Initialize ModelExplainer with Training Data

In [ ]:
# Load training data for explanations
print("\nLoading training data for explanations...")
X_train = load_training_data_for_explanation(sample_size=100)

if X_train is not None:
    print(f"✓ Training data loaded: {X_train.shape[0]} samples, {X_train.shape[1]} features")
    print(f"  Features: {', '.join(X_train.columns[:10])}...")
else:
    print("✗ Failed to load training data")
    X_train = None

In [ ]:
# Load trained model
print("Loading trained model...")
registry = ModelRegistry()

models = registry.list_available_models()
print(f"Available models: {models}")

if models:
    model_name = sorted(models)[0]
    model_path = registry.get_model_path(model_name)
    scaler_path = registry.get_scaler_path() if registry.scaler_exists() else None
    
    # Create prediction service
    service = PredictionService(model_path=model_path, scaler_path=scaler_path)
    
    if service.is_ready():
        print(f"✓ Model '{model_name}' loaded successfully")
        print(f"  Model path: {model_path}")
        print(f"  Scaler path: {scaler_path}")
    else:
        print("✗ Failed to load model")
else:
    print("✗ No models found. Please train a model first.")

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Any, List
import json

# Import XAI components
from src.ml_pipeline.explainability import ModelExplainer
from src.api.services import PredictionService, ModelRegistry, load_training_data_for_explanation
from src.api.schemas import HousePriceInput

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

## 1. Import Required Libraries and Load Model

# House Price Prediction - XAI (Explainable AI) Demonstration

This notebook demonstrates the complete XAI implementation for the house price prediction model, including SHAP, LIME, and feature importance analysis. It also shows how to test the `/explain` API endpoint on the backend.